# v2 同种子、逐样本四模型预测对比

本 notebook 用于比较同一训练种子下的 **FNO**、**FNO+$\gamma_{T}$**、**Oracle-Shared** 与 **Oracle-Shared（固定 $\rho=0$）**。它分别从训练集和测试集中选择任一样本，并在 $N=256$、$N=512$ 与 $N=1024$ 上使用各模型的验证集最佳 checkpoint 重新推理。

需要特别区分：$N=256$ 是训练分辨率；$N=512$ 和 $N=1024$ 是对同一连续样本重新解析采样后的零样本超分辨率预测，并非对 $N=256$ 数组做插值。

**图形契约：** 核心问题是四种模型对同一连续含间断样本的预测差异如何随分辨率变化；全域面板提供整体一致性证据，界面放大面板提供跳跃定位和局部振铃证据，指标表负责定量核对。图形属于双栏定量网格，主要审查风险是单样本选择偏差，因此个例图不替代全测试集统计。

## 1. 运行设备与路径

必须先设置可见 GPU，再导入 JAX。`PHYSICAL_GPU_INDEX` 使用服务器 `nvidia-smi` 显示的物理编号；按照项目约定，物理 GPU 6 不允许使用。如果只做 CPU 调试，可令 `REQUIRE_GPU=False` 并将 `PHYSICAL_GPU_INDEX=None`。

In [ ]:
import os
from pathlib import Path

PHYSICAL_GPU_INDEX = 0  # 按服务器实际空闲 GPU 修改；禁止设为 6
REQUIRE_GPU = True

if PHYSICAL_GPU_INDEX is not None:
    if int(PHYSICAL_GPU_INDEX) == 6:
        raise ValueError("按照 v2 方案，物理 GPU 6 被永久排除。")
    os.environ["CUDA_VISIBLE_DEVICES"] = str(PHYSICAL_GPU_INDEX)

cwd = Path.cwd().resolve()
root_candidates = [cwd, cwd / "deep_fno_shared_benchmark", *cwd.parents]
PROJECT_ROOT = next(
    (path for path in root_candidates if (path / "config.py").is_file() and (path / "models.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("无法定位 deep_fno_shared_benchmark 项目根目录。")
RESULTS_ROOT = PROJECT_ROOT / "results" / "formal_main_v2"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT.parent))

import jax
import pandas as pd
from IPython.display import display

from deep_fno_shared_benchmark.v2_sample_comparison import (
    MODEL_KINDS,
    build_sample_comparison,
    export_comparison,
    metrics_frame,
    plot_sample_comparison,
)

print("JAX backend:", jax.default_backend())
print("JAX devices:", jax.devices())
if REQUIRE_GPU and jax.default_backend() != "gpu":
    raise RuntimeError("REQUIRE_GPU=True，但当前 JAX backend 不是 GPU。")

## 2. 选择种子与样本

只需修改下列三个变量即可重复比较：

- `SEED`：正式训练种子，取 $0,1,2,3,4$；四种模型必须使用同一取值。
- `TRAIN_SAMPLE_INDEX`：训练集内部索引，范围为 $[0,15999]$。
- `TEST_SAMPLE_INDEX`：固定测试集内部索引，范围为 $[0,3999]$。

为了避免挑选性报告，论文主文若展示个例，建议事先固定样本选择规则，例如按 Oracle-Shared 与 FNO+$\gamma_{T}$ 的误差差值选取中位数样本；任意索引浏览更适合作为诊断。

In [ ]:
SEED = 0
TRAIN_SAMPLE_INDEX = 0
TEST_SAMPLE_INDEX = 0
RESOLUTIONS = (256, 512, 1024)
INTERFACE_HALF_WIDTH = 0.04
ZOOM_HALF_WIDTH = 0.12

if SEED not in range(5):
    raise ValueError("SEED 必须是 0、1、2、3 或 4。")
if not 0 <= TRAIN_SAMPLE_INDEX < 16_000:
    raise IndexError("TRAIN_SAMPLE_INDEX 必须位于 [0, 15999]。")
if not 0 <= TEST_SAMPLE_INDEX < 4_000:
    raise IndexError("TEST_SAMPLE_INDEX 必须位于 [0, 3999]。")

## 3. 训练集样本：同种子四模型对比

该部分重新生成种子 `SEED` 对应训练集中的指定连续样本，加载四种模型各自的验证集最佳 checkpoint，并分别在三个网格上前向推理。三行对应三种分辨率；左列显示全区域，右列放大终态界面 $\xi_{T}$ 附近。灰色点线为初值 $u_{0}$，黑线为解析真值 $u_{T}$。

In [ ]:
train_runs, train_comparisons = build_sample_comparison(
    results_root=RESULTS_ROOT,
    seed=SEED,
    split="train",
    sample_index=TRAIN_SAMPLE_INDEX,
    resolutions=RESOLUTIONS,
    interface_half_width=INTERFACE_HALF_WIDTH,
)

train_sample_id = train_comparisons[0].sample.sample_id
print(f"训练集样本：index={TRAIN_SAMPLE_INDEX}, sample_id={train_sample_id}")
for kind in MODEL_KINDS:
    run = train_runs[kind]
    print(f"{kind:>13}: {run.checkpoint_path}")

train_figure = plot_sample_comparison(train_comparisons, zoom_half_width=ZOOM_HALF_WIDTH)
display(train_figure)

### 训练样本定量指标

表中同时报告均方误差、相对 $L_{2}$ 误差、全域平均绝对误差、固定物理窗口内的界面平均绝对误差以及最大绝对误差。界面窗口定义为 $|x-\xi_{T}|\leq 0.04$，因此不同分辨率之间使用相同的物理宽度，而不是相同的网格点数。

In [ ]:
train_metrics = metrics_frame(train_comparisons)
display(
    train_metrics[[
        "resolution", "model", "mse", "relative_l2", "mae",
        "interface_mae", "max_abs_error"
    ]].style.format({
        "mse": "{:.4e}",
        "relative_l2": "{:.4e}",
        "mae": "{:.4e}",
        "interface_mae": "{:.4e}",
        "max_abs_error": "{:.4e}",
    })
)

## 4. 测试集样本：同种子四模型对比

测试集由固定随机种子生成，对五个训练种子及四种模型完全一致。这里仍然选择同一训练种子的四个最佳 checkpoint，以保证比较只改变模型结构，不改变训练重复编号。

In [ ]:
test_runs, test_comparisons = build_sample_comparison(
    results_root=RESULTS_ROOT,
    seed=SEED,
    split="test",
    sample_index=TEST_SAMPLE_INDEX,
    resolutions=RESOLUTIONS,
    interface_half_width=INTERFACE_HALF_WIDTH,
)

test_sample_id = test_comparisons[0].sample.sample_id
print(f"测试集样本：index={TEST_SAMPLE_INDEX}, sample_id={test_sample_id}")
test_figure = plot_sample_comparison(test_comparisons, zoom_half_width=ZOOM_HALF_WIDTH)
display(test_figure)

### 测试样本定量指标

逐样本指标用于解释图中局部差异，不能替代正式报告中的全测试集配对统计。若某一模型在单个样本上更优，只能说明该个例的行为；总体结论仍应以全部 $4000$ 个测试样本和五个训练种子的分层配对 bootstrap 为准。

In [ ]:
test_metrics = metrics_frame(test_comparisons)
display(
    test_metrics[[
        "resolution", "model", "mse", "relative_l2", "mae",
        "interface_mae", "max_abs_error"
    ]].style.format({
        "mse": "{:.4e}",
        "relative_l2": "{:.4e}",
        "mae": "{:.4e}",
        "interface_mae": "{:.4e}",
        "max_abs_error": "{:.4e}",
    })
)

## 5. 导出科研图与源数据

每幅图保存为含可编辑文字的 SVG、PDF、$600$ dpi TIFF 和 $400$ dpi PNG；每个图对应的逐网格点源数据及逐模型指标保存为 CSV，并额外生成图形 QA JSON。颜色采用色觉友好配色，并在所有面板中保持模型—颜色映射一致。导出文件名包含种子、数据划分、样本索引和 `sample_id`，便于审计。

In [ ]:
OUTPUT_DIR = RESULTS_ROOT / "comparison_four_models" / "samplewise_v2"
train_stem = f"train_seed{SEED}_index{TRAIN_SAMPLE_INDEX}_id{train_sample_id}"
test_stem = f"test_seed{SEED}_index{TEST_SAMPLE_INDEX}_id{test_sample_id}"

train_paths = export_comparison(
    train_comparisons, OUTPUT_DIR, stem=train_stem, figure=train_figure
)
test_paths = export_comparison(
    test_comparisons, OUTPUT_DIR, stem=test_stem, figure=test_figure
)

all_metrics = pd.concat([train_metrics, test_metrics], ignore_index=True)
combined_path = OUTPUT_DIR / f"selected_samples_seed{SEED}_metrics.csv"
all_metrics.to_csv(combined_path, index=False)

print("导出目录：", OUTPUT_DIR)
for name, path in {**{f"train_{k}": v for k, v in train_paths.items()},
                   **{f"test_{k}": v for k, v in test_paths.items()},
                   "combined_metrics_csv": combined_path}.items():
    print(f"{name:>28}: {path}")

## 6. 结果解读检查清单

1. 先检查四模型是否加载同一 `SEED` 的最佳 checkpoint，且 normalizer 完全一致。
2. 在 $N=256$ 比较训练分辨率内的拟合；在 $N=512$ 与 $N=1024$ 重点观察界面位置、跳跃两侧平台、振铃及远离界面的平滑背景。
3. 同时阅读全域图、界面放大图和误差表，避免仅凭曲线重叠程度下结论。
4. 个例图用于机制解释和异常诊断，不用于替代全测试集统计结论。